# Imports necessários

In [1]:
import pandas as pd 
import numpy as np
import json
import os
import time

from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel, ValidationError
from typing import Literal


# Obtenção dos dados

In [2]:
with open("../dados/dados_nivel_1.json", "r", encoding="utf-8") as f:
    dados = json.load(f)

print(dados.keys())

dict_keys(['taxa_cambio_usd_brl', 'operacoes'])


In [3]:
dados

{'taxa_cambio_usd_brl': 5.4,
 'operacoes': [{'id': 'OP-0001',
   'cliente_id': 'CLI-A-1',
   'data': '2026-03-09',
   'valor': 18100,
   'moeda': 'BRL',
   'canal': 'pix',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Alfa Comercio LTDA',
   'observacao': ''},
  {'id': 'OP-0002',
   'cliente_id': 'CLI-A-1',
   'data': '2026-03-09',
   'valor': 17300,
   'moeda': 'BRL',
   'canal': 'pix',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Alfa Comercio LTDA',
   'observacao': ''},
  {'id': 'OP-0003',
   'cliente_id': 'CLI-A-1',
   'data': '2026-03-09',
   'valor': 18800,
   'moeda': 'BRL',
   'canal': 'ted',
   'tipo': 'transferencia_enviada',
   'contraparte': 'Beta Servicos ME',
   'observacao': ''},
  {'id': 'OP-0004',
   'cliente_id': 'CLI-A-1',
   'data': '2026-03-21',
   'valor': 3300,
   'moeda': 'BRL',
   'canal': 'boleto',
   'tipo': 'pagamento',
   'contraparte': 'Gama Distribuidora',
   'observacao': ''},
  {'id': 'OP-0005',
   'cliente_id': 'CLI-A-2',
   'data':

In [4]:
# "taxa_cambio_usd_brl" é um campo único
list(dados.keys()).count("taxa_cambio_usd_brl")

1

In [5]:
TAXA_CAMBIO = float(dados["taxa_cambio_usd_brl"])
operacoes = dados["operacoes"]

* Em 'operacoes', todos os registros possuem a mesma estrutura?

In [6]:
def validar_estrutura_operacoes(operacoes):
    if not operacoes:
        raise ValueError("A lista de operações está vazia.")

    chaves_esperadas = set(operacoes[0].keys())
    inconsistencias = []

    for i, operacao in enumerate(operacoes):
        chaves = set(operacao.keys())

        if chaves != chaves_esperadas:
            inconsistencias.append({
                "indice": i,
                "faltando": sorted(chaves_esperadas - chaves),
                "a_mais": sorted(chaves - chaves_esperadas)
            })

    return {
        "valido": len(inconsistencias) == 0,
        "chaves_esperadas": chaves_esperadas,
        "inconsistencias": inconsistencias
    }

In [7]:
resultado = validar_estrutura_operacoes(operacoes)

resultado

{'valido': True,
 'chaves_esperadas': {'canal',
  'cliente_id',
  'contraparte',
  'data',
  'id',
  'moeda',
  'observacao',
  'tipo',
  'valor'},
 'inconsistencias': []}

#### Todos os registros possuem as mesmas chaves, portanto, podemos criar o DataFrame somente com 'operacoes'.

In [8]:
df = pd.DataFrame(operacoes)

# como é um dataframe pequeno, prefiro visualizá-lo todo.
df

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


#### Pontos de qualidade dos dados:
* dados nos formatos adequados? existem Nans? existem duplicatas?
* ID's são únicos?
* datas estão todas no mesmo formato? são todas datas válidas?
* existem dados diferentes que representam a mesma coisa? por exemplo: pix e PIX, deposito e depósito...


# Entendimento da Base

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           20 non-null     object
 1   cliente_id   20 non-null     object
 2   data         19 non-null     object
 3   valor        20 non-null     int64 
 4   moeda        20 non-null     object
 5   canal        20 non-null     object
 6   tipo         20 non-null     object
 7   contraparte  20 non-null     object
 8   observacao   20 non-null     object
dtypes: int64(1), object(8)
memory usage: 1.5+ KB


In [10]:
df[df.duplicated(keep=False)]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [11]:
df[df['data'].isna()]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,None,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


* há um registro duplicado de id = OP-0007 --> como todos os dados são os mesmos, removeremos a duplicata;
* existe uma data Nan, com observação de 'data nao capturada pelo sistema'. o que fazer com esse registro? mantemos, mas ficará sob nosso radar;
* 'valor' está como inteiro, mas podemos transforma-lo para float;
* a seguir vamos fazer as demais verificações/limpezas na base.

In [12]:
df = df.drop_duplicates(keep="first")

In [13]:
df['valor'] = df['valor'].astype(float)

C:\Users\yghor\AppData\Local\Temp\ipykernel_17052\1997309100.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['valor'] = df['valor'].astype(float)


In [14]:
df.shape

(19, 9)

## id

In [15]:
# 19 id's únicos, assim como 19 registros na base.
df['id'].nunique()

19

## cliente_id

In [16]:
# seis clientes únicos
df['cliente_id'].value_counts()

cliente_id
CLI-A-1    4
CLI-A-4    4
CLI-A-5    4
CLI-A-3    3
CLI-A-2    2
CLI-A-6    2
Name: count, dtype: int64

## data

In [17]:
# as datas estão formatadas todas da mesma forma? Não, há um registro com data = None --> data nao capturada pelo sistema
# o cliente dessa operação é um dos com maior volume de dados na nossa base
df[~df["data"].str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,None,4300.0,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


In [18]:
# todas são datas válidas? todas as datas, com exceção do caso encontrado, são data válidas!
datas = pd.to_datetime(df["data"], format="%Y-%m-%d", errors="coerce")

df[datas.isna()]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,None,4300.0,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


In [19]:
df["data"] = pd.to_datetime(df["data"])

C:\Users\yghor\AppData\Local\Temp\ipykernel_17052\1343936737.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["data"] = pd.to_datetime(df["data"])


## valor

In [20]:
# não há valores nulos
df[df['valor'].isna()]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao


## moeda

In [21]:
# somente um registro em USD
df['moeda'].value_counts()

moeda
BRL    18
USD     1
Name: count, dtype: int64

## valor_brl

In [22]:
# criamos uma coluna para normalizar valores para BRL
df["valor_brl"] = df["valor"]

df.loc[df["moeda"] == "USD", "valor_brl"] = (
    df.loc[df["moeda"] == "USD", "valor"] * TAXA_CAMBIO
)

C:\Users\yghor\AppData\Local\Temp\ipykernel_17052\184986511.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["valor_brl"] = df["valor"]


## canal

In [23]:
# não há valores anormais
df['canal'].value_counts()

canal
pix        8
ted        5
boleto     3
cartao     2
especie    1
Name: count, dtype: int64

## tipo

In [24]:
# não há valores anormais
df['tipo'].value_counts()

tipo
transferencia_enviada     10
pagamento                  5
transferencia_recebida     3
deposito                   1
Name: count, dtype: int64

## contraparte	

In [25]:
# não há valores anormais
df['contraparte'].value_counts()

contraparte
Alfa Comercio LTDA     4
Delta Transportes      4
Beta Servicos ME       3
Gama Distribuidora     3
Epsilon Consultoria    3
Zeta Importacao        2
Name: count, dtype: int64

# Investigações

## Volume por cliente

In [26]:
volume_cliente = (
    df.groupby("cliente_id")["valor_brl"]
      .sum()
      .reset_index(name="volume_total_brl")
)

volume_cliente

,cliente_id,volume_total_brl
0,CLI-A-1,57500.0
1,CLI-A-2,52900.0
2,CLI-A-3,48500.0
3,CLI-A-4,79500.0
4,CLI-A-5,16900.0
5,CLI-A-6,10200.0


## Operações por Canal

In [27]:
# essa informação deveria ser feita por cliente?

operacoes_canal = (
    df.groupby("canal")
      .size()
      .reset_index(name="quantidade_operacoes")
)

operacoes_canal

,canal,quantidade_operacoes
0,boleto,3
1,cartao,2
2,especie,1
3,pix,8
4,ted,5


## Regra 1 — Fracionamento

In [28]:
# Precisamos saber quantidade de operações, soma dos valores e maior operação para cada par cliente-data
fracionamento = (
    df.groupby(["cliente_id", "data"])
      .agg(
          quantidade_operacoes=("id", "count"),
          soma_valor_brl=("valor_brl", "sum"),
          maior_operacao_brl=("valor_brl", "max")
      )
      .reset_index()
)

# Aplicando a regra
fracionamento["fracionamento"] = (
    (fracionamento["quantidade_operacoes"] >= 3) &
    (fracionamento["soma_valor_brl"] > 50_000) &
    (fracionamento["maior_operacao_brl"] < 20_000)
)

fracionamento

,cliente_id,data,quantidade_operacoes,soma_valor_brl,maior_operacao_brl,fracionamento
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True
1,CLI-A-1,2026-03-21,1,3300.0,3300.0,False
2,CLI-A-2,2026-03-14,2,52900.0,27000.0,False
3,CLI-A-3,2026-03-05,3,48500.0,17200.0,False
4,CLI-A-4,2026-03-03,1,3800.0,3800.0,False
5,CLI-A-4,2026-03-11,1,5100.0,5100.0,False
6,CLI-A-4,2026-03-18,1,5800.0,5800.0,False
7,CLI-A-4,2026-03-24,1,64800.0,64800.0,False
8,CLI-A-5,2026-03-07,1,2900.0,2900.0,False
9,CLI-A-5,2026-03-16,1,7000.0,7000.0,False


O cliente CLI-A-5 possui uma transação sem data, porém, mesmo incluindo esta na agregação, o mesmo não entraria nos critérios de flag, visto que possui no máximo uma transação por data.

Agora vamos adicionar as flags no df original:

In [29]:
clientes_fracionamento = (
    fracionamento.loc[
        fracionamento["fracionamento"],
        "cliente_id"
    ]
    .unique()
)

df["flag_fracionamento"] = (
    df["cliente_id"].isin(clientes_fracionamento)
)

C:\Users\yghor\AppData\Local\Temp\ipykernel_17052\787473797.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["flag_fracionamento"] = (


#### Tomemos como comparação os casos:
* CLI-A-3 em 2026-03-05
* CLI-A-1 em 2026-03-09

Ambos possuem três operações, mas somente o segundo foi marcado. Apesar do primeiro também ter a maior operação até 20000, a soma dos valores das transações não ultrapassa 50000.

In [30]:
casos = fracionamento[
    (
        (fracionamento["cliente_id"] == "CLI-A-3") &
        (fracionamento["data"] == "2026-03-05")
    )
    |
    (
        (fracionamento["cliente_id"] == "CLI-A-1") &
        (fracionamento["data"] == "2026-03-09")
    )
]

casos

,cliente_id,data,quantidade_operacoes,soma_valor_brl,maior_operacao_brl,fracionamento
0,CLI-A-1,2026-03-09,3,54200.0,18800.0,True
3,CLI-A-3,2026-03-05,3,48500.0,17200.0,False


## Regra 2 - Valor atípico

In [31]:
estatisticas_cliente = (
    df.groupby("cliente_id")["valor_brl"]
      .agg(
          quantidade_operacoes="count",
          mediana_brl="median"
      )
      .reset_index()
)

In [32]:
estatisticas_cliente

,cliente_id,quantidade_operacoes,mediana_brl
0,CLI-A-1,4,17700.0
1,CLI-A-2,2,26450.0
2,CLI-A-3,3,16100.0
3,CLI-A-4,4,5450.0
4,CLI-A-5,4,3600.0
5,CLI-A-6,2,5100.0


In [33]:
estatisticas_cliente["5x_mediana_brl"] = 5 * estatisticas_cliente["mediana_brl"]

In [34]:
df = df.merge(
    estatisticas_cliente,
    on="cliente_id",
    how="left"
)

In [35]:
df[['cliente_id', 'quantidade_operacoes', 'valor_brl', 'mediana_brl', '5x_mediana_brl']].sample(5)

,cliente_id,quantidade_operacoes,valor_brl,mediana_brl,5x_mediana_brl
2,CLI-A-1,4,18800.0,17700.0,88500.0
16,CLI-A-5,4,4300.0,3600.0,18000.0
10,CLI-A-4,4,5100.0,5450.0,27250.0
1,CLI-A-1,4,17300.0,17700.0,88500.0
8,CLI-A-3,3,16100.0,16100.0,80500.0


In [36]:
df["flag_valor_atipico"] = (
    (df["quantidade_operacoes"] >= 4) &
    (df["valor_brl"] > df["5x_mediana_brl"])
)

In [37]:
df.loc[
    df["flag_valor_atipico"],
    [
        "id",
        "cliente_id",
        "quantidade_operacoes",
        "mediana_brl",
        "5x_mediana_brl",
        "valor_brl",
        "flag_valor_atipico"
    ]
]

,id,cliente_id,quantidade_operacoes,mediana_brl,5x_mediana_brl,valor_brl,flag_valor_atipico
12,OP-0013,CLI-A-4,4,5450.0,27250.0,64800.0,True


#### Comparando CLI-A-4 e CLI-A-5, pois ambos possuem ao menos 4 operações:
* apenas a operação de CLI-A-4 em 2026-03-24 supera em cinco vezes a mediana do valores transacionados, portanto, é o único registro com a flag True.

In [38]:
clientes_validacao = ["CLI-A-4", "CLI-A-5"]

validacao = df[
    df["cliente_id"].isin(clientes_validacao)
][[
    "id",
    "cliente_id",
    "data",
    "quantidade_operacoes",
    "mediana_brl",
    "5x_mediana_brl",
    "valor_brl",
    "flag_valor_atipico"
]].sort_values(["cliente_id", "valor_brl"])

validacao

,id,cliente_id,data,quantidade_operacoes,mediana_brl,5x_mediana_brl,valor_brl,flag_valor_atipico
9,OP-0010,CLI-A-4,2026-03-03,4,5450.0,27250.0,3800.0,False
10,OP-0011,CLI-A-4,2026-03-11,4,5450.0,27250.0,5100.0,False
11,OP-0012,CLI-A-4,2026-03-18,4,5450.0,27250.0,5800.0,False
12,OP-0013,CLI-A-4,2026-03-24,4,5450.0,27250.0,64800.0,True
15,OP-0016,CLI-A-5,2026-03-26,4,3600.0,18000.0,2700.0,False
13,OP-0014,CLI-A-5,2026-03-07,4,3600.0,18000.0,2900.0,False
16,OP-0017,CLI-A-5,NaT,4,3600.0,18000.0,4300.0,False
14,OP-0015,CLI-A-5,2026-03-16,4,3600.0,18000.0,7000.0,False


# Análise com LLM

## Configurações 

In [41]:
load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

## Funções

In [42]:
def cria_resumo_cliente(df, cliente_id):
    df_cliente = df.loc[df["cliente_id"] == cliente_id].copy()

    if df_cliente.empty:
        raise ValueError(f"Cliente {cliente_id} não encontrado.")

    # Garantir ordenação temporal
    df_cliente["data"] = pd.to_datetime(df_cliente["data"])
    df_cliente = df_cliente.sort_values("data") 

    intervalos = (
            df_cliente["data"]
            .diff()
            .dt.days
            .dropna()
        )

    operacoes_por_data = (
            df_cliente
            .groupby("data")
            .size()
        )


    resumo_cliente = {
        # Indicadores gerais
        "quantidade_operacoes": len(df_cliente),
        "volume_total_brl": df_cliente["valor_brl"].sum(),
        "mediana_valor_brl": df_cliente["valor_brl"].median(),

        # Resultados das regras
        "flag_fracionamento": df_cliente["flag_fracionamento"].any(),
        "quantidade_valor_atipico": (
            df_cliente["flag_valor_atipico"].sum()
        ),

        # Diversidade operacional
        "n_canais": df_cliente["canal"].nunique(),
        "n_tipos_operacao": df_cliente["tipo"].nunique(),
        "n_contrapartes": df_cliente["contraparte"].nunique(),

        # Comportamento temporal
        "menor_intervalo_dias": (
            intervalos.min()
            if not intervalos.empty
            else None
        ),

        "maior_quantidade_operacoes_mesma_data": (
            operacoes_por_data.max()
        ),
    }

    return resumo_cliente

In [43]:
def cria_prompt_1(resumo_cliente):
    return f"""
                Analise o comportamento do cliente abaixo com base exclusivamente nos indicadores fornecidos, 
                e produza um parecer sobre o comportamento do cliente em relação à lavagem de dinheiro.

                IMPORTANTE:
                - Os indicadores foram calculados previamente por um sistema determinístico.
                - NÃO faça novos cálculos.
                - NÃO altere os valores fornecidos.
                - NÃO invente informações que não estejam no contexto.
                - NÃO compare valores.
                - NÃO sugira cálculos adicionais, dados faltantes ou próximos passos fora do que foi pedido.
                - Sua função é interpretar os indicadores e redigir um parecer.

                Quantidade de operações: {resumo_cliente["quantidade_operacoes"]}
                Volume total (BRL): {resumo_cliente["volume_total_brl"]}
                Possui flag de fracionamento: {resumo_cliente["flag_fracionamento"]}
                Quantidade de valores atípicos: {resumo_cliente["quantidade_valor_atipico"]}

                Retorne SOMENTE um JSON válido, sem markdown ou texto adicional,
                com exatamente os seguintes campos:

                {{
                    "nivel_risco": "baixo|médio|alto",
                    "tipologia_suspeita": "possível tipologia ou ausência de tipologia evidente",
                    "red_flags": ["sinal 1", "sinal 2", ...],
                    "justificativa": "justificativa objetiva da classificação"
                }}
            """

In [44]:
def cria_prompt_2(resumo_cliente):
    return f"""
                Você é um analista de Prevenção à Lavagem de Dinheiro (PLD) de um banco.

                Analise o comportamento do cliente abaixo com base exclusivamente
                nos indicadores fornecidos e produza um parecer sobre o comportamento
                do cliente.

                IMPORTANTE:
                - Todos os indicadores foram calculados previamente por um sistema
                determinístico.
                - NÃO faça novos cálculos.
                - NÃO altere os valores fornecidos.
                - NÃO invente informações que não estejam no contexto.
                - NÃO compare valores.
                - NÃO tente reconstruir ou validar as regras determinísticas.
                - NÃO sugira cálculos adicionais, dados faltantes ou próximos passos.
                - Sua função é interpretar os padrões apresentados e redigir um parecer.

                INDICADORES GERAIS

                Quantidade de operações:
                {resumo_cliente["quantidade_operacoes"]}

                Volume total (BRL):
                {resumo_cliente["volume_total_brl"]}

                Mediana dos valores (BRL):
                {resumo_cliente["mediana_valor_brl"]}

                RESULTADOS DAS REGRAS

                Possui flag de fracionamento:
                {resumo_cliente["flag_fracionamento"]}

                Quantidade de valores atípicos:
                {resumo_cliente["quantidade_valor_atipico"]}

                DIVERSIDADE OPERACIONAL

                Número de canais distintos:
                {resumo_cliente["n_canais"]}

                Número de tipos de operação distintos:
                {resumo_cliente["n_tipos_operacao"]}

                Número de contrapartes distintas:
                {resumo_cliente["n_contrapartes"]}

                COMPORTAMENTO TEMPORAL

                Maior quantidade de operações realizadas na mesma data:
                {resumo_cliente["maior_quantidade_operacoes_mesma_data"]}

                Com base exclusivamente nessas informações, produza um parecer
                sobre o comportamento do cliente.

                Retorne SOMENTE um JSON válido, sem markdown ou texto adicional,
                com exatamente os seguintes campos:

                {{
                    "nivel_risco": "baixo|médio|alto",
                    "tipologia_suspeita": "possível tipologia ou ausência de tipologia evidente",
                    "red_flags": ["sinal 1", "sinal 2", ...],
                    "justificativa": "justificativa objetiva da classificação"
                }}
            """

In [45]:
def consultar_llm(prompt):
    inicio = time.perf_counter()

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    tempo_resposta = time.perf_counter() - inicio

    usage = response.usage_metadata

    return {
        "resposta": response.text,
        "tempo_segundos": tempo_resposta,
        "tokens_entrada": usage.prompt_token_count,
        "tokens_saida": usage.candidates_token_count,
        "tokens_raciocinio": usage.thoughts_token_count or 0,
        "tokens_total": usage.total_token_count
    }

In [46]:
class Parecer(BaseModel):
    nivel_risco: Literal["baixo", "médio", "alto"]
    tipologia_suspeita: str
    red_flags: list[str]
    justificativa: str

def validar_parecer(resposta):
    try:
        dados = json.loads(resposta)
        parecer = Parecer.model_validate(dados)

        return parecer, None

    except json.JSONDecodeError as erro:    
        return None, f"JSON inválido: {erro}"

    except ValidationError as erro:
        return None, f"Schema inválido: {erro}"

In [47]:
def gerar_parecer(prompt, max_tentativas=2):
    resultados = []

    prompt_atual = prompt

    for tentativa in range(1, max_tentativas + 1):

        resultado_llm = consultar_llm(prompt_atual)

        parecer, erro = validar_parecer(
            resultado_llm["resposta"]
        )

        resultados.append({
            "tentativa": tentativa,
            **resultado_llm,
            "valido": parecer is not None,
            "erro": erro
        })

        if parecer is not None:
            return {
                "parecer": parecer,
                "tentativas": resultados
            }

        # Se falhou, prepara uma nova tentativa
        prompt_atual = f"""
                            A resposta anterior não respeitou o formato solicitado.

                            Erro encontrado:
                            {erro}

                            Corrija SOMENTE o formato da resposta.

                            Retorne SOMENTE um JSON válido, sem markdown e sem texto adicional,
                            contendo exatamente:

                            {{
                                "nivel_risco": "baixo|médio|alto",
                                "tipologia_suspeita": "string",
                                "red_flags": ["string"],
                                "justificativa": "string"
                            }}
                        """

    return {
        "parecer": None,
        "tentativas": resultados
    }

In [48]:
def analisar_cliente(df, cliente_id):
    # ============================================================
    # 1. Criar resumo determinístico com pandas
    # ============================================================

    resumo_cliente = cria_resumo_cliente(
        df=df,
        cliente_id=cliente_id
    )

    # ============================================================
    # 2. Criar os dois prompts
    # ============================================================

    prompt_1 = cria_prompt_1(
        resumo_cliente=resumo_cliente
    )

    prompt_2 = cria_prompt_2(
        resumo_cliente=resumo_cliente
    )

    # ============================================================
    # 3. Executar os dois prompts
    # ============================================================

    resultado_1 = gerar_parecer(prompt_1)

    resultado_2 = gerar_parecer(prompt_2)

    # ============================================================
    # 4. Comparar os resultados
    # ============================================================

    comparacao = pd.DataFrame([
        {
            "prompt": "Prompt 1",
            "tempo_s": sum(
                t["tempo_segundos"]
                for t in resultado_1["tentativas"]
            ),
            "tokens_entrada": sum(
                t["tokens_entrada"]
                for t in resultado_1["tentativas"]
            ),
            "tokens_saida": sum(
                t["tokens_saida"]
                for t in resultado_1["tentativas"]
            ),
            "tokens_total": sum(
                t["tokens_total"]
                for t in resultado_1["tentativas"]
            ),
            "nivel_risco": (
                resultado_1["parecer"].nivel_risco
                if resultado_1["parecer"]
                else None
            ),
        },
        {
            "prompt": "Prompt 2",
            "tempo_s": sum(
                t["tempo_segundos"]
                for t in resultado_2["tentativas"]
            ),
            "tokens_entrada": sum(
                t["tokens_entrada"]
                for t in resultado_2["tentativas"]
            ),
            "tokens_saida": sum(
                t["tokens_saida"]
                for t in resultado_2["tentativas"]
            ),
            "tokens_total": sum(
                t["tokens_total"]
                for t in resultado_2["tentativas"]
            ),
            "nivel_risco": (
                resultado_2["parecer"].nivel_risco
                if resultado_2["parecer"]
                else None
            ),
        }
    ])

    # ============================================================
    # 5. Registrar todas as tentativas
    # ============================================================

    tentativas = []

    for resultado, nome_prompt in [
        (resultado_1, "Prompt 1"),
        (resultado_2, "Prompt 2")
    ]:
        for tentativa in resultado["tentativas"]:
            tentativas.append({
                "cliente_id": cliente_id,
                "prompt": nome_prompt,
                **tentativa
            })

    df_tentativas = pd.DataFrame(tentativas)

    # ============================================================
    # 6. Retornar tudo
    # ============================================================

    return {
        "cliente_id": cliente_id,
        "resumo_cliente": resumo_cliente,
        "prompt_1": prompt_1,
        "prompt_2": prompt_2,
        "resultado_1": resultado_1,
        "resultado_2": resultado_2,
        "comparacao": comparacao,
        "df_tentativas": df_tentativas
    }

O cliente CLI-A-1 foi escolhido por ter flag_fracionamento.

In [49]:
cli_a_1 = analisar_cliente(df, 'CLI-A-1')

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Resultado da chamada usando o Prompt 1

In [50]:
cli_a_1["resultado_1"]["parecer"].model_dump()

{'nivel_risco': 'médio',
 'tipologia_suspeita': 'Fracionamento de operações (Smurfing)',
 'red_flags': ['Presença de flag de fracionamento'],
 'justificativa': 'O cliente realizou 4 operações com volume total de 57500.0 BRL e apresenta a flag de fracionamento ativada, o que aponta para um comportamento de potencial divisão de recursos, embora a quantidade de valores atípicos seja igual a 0.'}

Resultado da chamada usando o Prompt 2

In [51]:
cli_a_1["resultado_2"]["parecer"].model_dump()

{'nivel_risco': 'alto',
 'tipologia_suspeita': 'Fracionamento (Smurfing)',
 'red_flags': ['Presença de flag de fracionamento de operações',
  'Alta concentração de operações na mesma data',
  'Utilização de múltiplos canais e contrapartes distintas'],
 'justificativa': 'O cliente apresentou flag positiva para fracionamento de operações, realizando 3 de suas 4 movimentações em uma mesma data. A conduta envolveu um volume total de R$ 57.500,00 distribuído entre 3 canais e 3 contrapartes distintas, configurando padrão típico de tentativa de pulverização ou divisão de valores para burlar limites operacionais.'}

Teste de resposta malformada: Para validar o tratamento de erros sem depender de uma falha real da API, foi simulada uma resposta inválida contendo um tipo incorreto em red_flags e ausência do campo justificativa. A função validar_parecer() rejeitou a resposta e retornou o erro, impedindo que um parecer inválido fosse utilizado.

In [52]:
resposta_malformada = """
{
    "nivel_risco": "alto",
    "tipologia_suspeita": "Fracionamento",
    "red_flags": "flag de fracionamento"
}
"""

resultado = validar_parecer(resposta_malformada)

print(resultado)

(None, "Schema inválido: 2 validation errors for Parecer\nred_flags\n  Input should be a valid list [type=list_type, input_value='flag de fracionamento', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/list_type\njustificativa\n  Field required [type=missing, input_value={'nivel_risco': 'alto', '...'flag de fracionamento'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing")


Comparação entre os desempenhos e resultados dos dois prompts

In [53]:
cli_a_1["comparacao"]

,prompt,tempo_s,tokens_entrada,tokens_saida,tokens_total,nivel_risco
0,Prompt 1,7.536517,291,128,1593,médio
1,Prompt 2,8.523825,450,174,1866,alto


O Prompt 1, com indicadores mais agregados, classificou o cliente como risco médio e identificou apenas a flag de fracionamento. Já o Prompt 2, com informações adicionais sobre concentração temporal, canais e contrapartes, classificou como alto risco e identificou mais red flags, produzindo uma justificativa mais específica.

Essa maior contextualização teve um custo: o Prompt 2 utilizou 450 tokens de entrada contra 291 e 174 de saída contra 128, além de apresentar maior tempo de resposta (8,52s contra 7,54s). Assim, o Prompt 2 trouxe uma análise mais detalhada e sensível ao comportamento, enquanto o Prompt 1 foi mais simples, rápido e econômico.